# BL2403 — Phase 4-6: Model Analysis & Pruning

This notebook covers:
1. Loading pre-built graphs and dataset splits
2. Training / loading a GNN model
3. Evaluation on the test set (accuracy, F1, ROC-AUC)
4. Graph pruning experiments (attention, degree, random)
5. Performance comparison before / after pruning

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.config import DEVICE, CHECKPOINT_DIR, PLOT_DIR, RESULTS_DIR
from src.dataset import load_splits
from src.models import get_model
from src.train import evaluate, train_model, plot_roc_curves
from src.pruning import run_pruning_experiment
from torch_geometric.loader import DataLoader

print(f'Using device: {DEVICE}')

## 1. Load Dataset Splits

In [ ]:
train_graphs, val_graphs, test_graphs = load_splits()
in_channels = train_graphs[0].x.size(1)
print(f'Train: {len(train_graphs)}  Val: {len(val_graphs)}  Test: {len(test_graphs)}')
print(f'Node feature dim: {in_channels}')
print(f'Nodes per graph: {train_graphs[0].x.size(0)}')
print(f'Edges per graph: {train_graphs[0].edge_index.size(1)}')

## 2. Train a GAT Model (or Load Checkpoint)

In [ ]:
MODEL_NAME = 'gat'   # change to 'gcn', 'sage', or 'gin' as needed

model = get_model(
    MODEL_NAME,
    in_channels=in_channels,
    hidden_dim=128,
    num_layers=3,
    num_classes=3,
    dropout=0.3,
    pooling='mean',
).to(DEVICE)

ckpt_path = os.path.join(CHECKPOINT_DIR, f'{MODEL_NAME}_best_val_f1.pt')

if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    print(f'Loaded checkpoint: {ckpt_path}')
else:
    print('No checkpoint found — training from scratch …')
    history = train_model(
        model=model,
        train_graphs=train_graphs,
        val_graphs=val_graphs,
        model_name=MODEL_NAME,
        max_epochs=100,
        patience=15,
    )
    # reload best weights
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

## 3. Test Set Evaluation

In [ ]:
import torch.nn as nn
criterion = nn.CrossEntropyLoss()
test_loader = DataLoader(test_graphs, batch_size=32, shuffle=False)

test_loss, test_metrics = evaluate(model, test_loader, criterion)
print(f'Test Loss: {test_loss:.4f}')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Classification report
from sklearn.metrics import classification_report

all_true, all_pred = [], []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        logits = model(batch.x, batch.edge_index, batch.batch)
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_true.extend(batch.y.cpu().numpy())
        all_pred.extend(preds)

print(classification_report(
    all_true, all_pred,
    target_names=['Short (0)', 'Mid (1)', 'Long (2)'],
    zero_division=0,
))

## 4. ROC Curves

In [ ]:
plot_roc_curves(model, test_graphs, MODEL_NAME)

## 5. Training Curves (if available)

In [ ]:
loss_img = os.path.join(PLOT_DIR, f'{MODEL_NAME}_loss.png')
f1_img   = os.path.join(PLOT_DIR, f'{MODEL_NAME}_f1.png')

from IPython.display import Image, display
if os.path.exists(loss_img):
    display(Image(loss_img))
if os.path.exists(f1_img):
    display(Image(f1_img))

## 6. Graph Pruning Experiment

In [ ]:
pruning_results = run_pruning_experiment(
    model, test_graphs, keep_ratios=(0.25, 0.50, 0.75)
)

In [ ]:
import pandas as pd

results_df = pd.DataFrame(pruning_results).T.reset_index()
results_df.columns = ['Strategy', 'Accuracy', 'F1 (Macro)', 'Total ms', 'ms/graph']
results_df = results_df.sort_values('F1 (Macro)', ascending=False)
results_df.style.highlight_max(subset=['Accuracy', 'F1 (Macro)'], color='lightgreen') \
                .highlight_min(subset=['ms/graph'], color='lightblue')

In [ ]:
# Bar chart: F1 comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.barh(results_df['Strategy'], results_df['F1 (Macro)'], color='steelblue', alpha=0.8)
ax.set_xlabel('Macro F1')
ax.set_title('Pruning Strategies – Macro F1')
ax.axvline(pruning_results['baseline']['f1_macro'], color='red',
           linestyle='--', label='Baseline')
ax.legend()

ax = axes[1]
ax.barh(results_df['Strategy'], results_df['ms/graph'], color='darkorange', alpha=0.8)
ax.set_xlabel('Inference Time (ms/graph)')
ax.set_title('Pruning Strategies – Inference Speed')
ax.axvline(pruning_results['baseline']['ms_per_graph'], color='red',
           linestyle='--', label='Baseline')
ax.legend()

plt.tight_layout()
plt.show()